In [3]:
!pip install optuna

In [4]:
import xgboost as xgb
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

def df_Data_Cleansing(df):
    cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().astype('category')

    df['short_sleep_flag'] = (df['sleep_duration'] < 6.0).astype(int)
    df["exercise_strength"] = df['calorie_expenditure'] / ((df['exercise_duration'] + 0.1) * df['bmi'])

    df['healthy_score'] = (
        df['sleep_duration'].between(7, 8).astype(int) +
        (df['bmi'] <= 19).astype(int) +
        (df['exercise_duration'] >= 40).astype(int) +
        (df['stress_level'] == 'low').astype(int) +
        (df['sleep_quality'] == 'good').astype(int) +
        (df['physical_activity_level'] == 'active').astype(int)
    )

    df['unhealthy_score'] = (
        df['sleep_duration'].between(5, 6).astype(int) +
        (df['bmi'] >= 27).astype(int) +
        (df['calorie_expenditure'] <= 1400).astype(int) +
        (df['step_count'] <= 5000).astype(int) +
        (df['exercise_duration'] <= 30).astype(int) +
        (df['stress_level'] == 'high').astype(int) +
        (df['sleep_quality'] == 'poor').astype(int) +
        df['physical_activity_level'].isin(['sedentary', 'moderate']).astype(int) +
        (df['smoking_alcohol'] == 'yes').astype(int)
    )

    stress_mapping = {'low': 1, 'medium': 2, 'high': 3}
    df['stress_level_num'] = df['stress_level'].map(stress_mapping).astype(float)
    df['stress_per_sleep'] = df['stress_level_num'] / df['sleep_duration']

    drop_cols = [
        'stress_level_num',
        'diet_type',
        'gender',
        'smoking_alcohol',
    ]
    df = df.drop(columns=drop_cols, errors='ignore')

    return df

df = df_Data_Cleansing(pd.read_csv('data/train.csv', encoding="utf-8"))

X = df.drop(['id', 'health_condition'], axis=1)
y = df['health_condition']

le_stress = LabelEncoder()
X['stress_level'] = le_stress.fit_transform(X['stress_level'])

NUMS = X.select_dtypes(include="float").columns.tolist() + X.select_dtypes(include="int").columns.tolist()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

def objective(trial):
    params = {
        'objective': 'multi:softmax',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'enable_categorical': True,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in cv.split(X, y_encoded):
        X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_valid = y_encoded[train_idx], y_encoded[valid_idx]

        train_medians = X_train[NUMS].median()
        X_train[NUMS] = X_train[NUMS].fillna(train_medians)
        X_valid[NUMS] = X_valid[NUMS].fillna(train_medians)

        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        model = xgb.XGBClassifier(**params, n_estimators=1000, early_stopping_rounds=50)

        model.fit(
            X_train, y_train,
            sample_weight=sample_weights,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        preds = model.predict(X_valid)
        score = balanced_accuracy_score(y_valid, preds)
        scores.append(score)

    return np.mean(scores)

print("Optunaによるパラメータ探索を開始します")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("ベストスコア:", study.best_value)
print("ベストパラメータ:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

[I 2026-07-21 14:43:08,158] A new study created in memory with name: no-name-dc16fa31-ca9c-48a3-a215-c765f0cff625


Optunaによるパラメータ探索を開始します


[I 2026-07-21 14:48:39,676] Trial 0 finished with value: 0.9486969795490964 and parameters: {'learning_rate': 0.007837763430894595, 'max_depth': 9, 'min_child_weight': 9, 'subsample': 0.6283130633985872, 'colsample_bytree': 0.781468094970178}. Best is trial 0 with value: 0.9486969795490964.
[I 2026-07-21 14:52:51,218] Trial 1 finished with value: 0.9459793308363806 and parameters: {'learning_rate': 0.026913546266729507, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.720443347390227, 'colsample_bytree': 0.9273478342882497}. Best is trial 0 with value: 0.9486969795490964.
[I 2026-07-21 14:58:33,252] Trial 2 finished with value: 0.9485892268176315 and parameters: {'learning_rate': 0.007917759614580718, 'max_depth': 10, 'min_child_weight': 10, 'subsample': 0.5526073070872012, 'colsample_bytree': 0.8433123134997096}. Best is trial 0 with value: 0.9486969795490964.
[I 2026-07-21 15:03:36,182] Trial 3 finished with value: 0.9480831609627245 and parameters: {'learning_rate': 0.010004945

ベストスコア: 0.94940639303098
ベストパラメータ:
    learning_rate: 0.029880659828276236
    max_depth: 5
    min_child_weight: 9
    subsample: 0.9527064742344795
    colsample_bytree: 0.8672601416983036
